# 野生蓝莓产量预测分析## 步骤 4：线性回归

In [ ]:
import pandas as pdimport numpy as npimport sys, ossys.path.insert(0, os.path.join(os.getcwd(), ".."))from src.config import *from src.data.loader import load_trainfrom src.data.preprocessor import DataPreprocessorfrom src.models.linear_regression import BlueberryLinearRegression, compute_viffrom src.features.engineering import apply_pcafrom src.visualization.plots import *train = load_train()preprocessor = DataPreprocessor()df_clean = preprocessor.clean(train)X, y = preprocessor.split_features_target(df_clean)X_scaled = preprocessor.scale(X, fit=True)X_train, X_val, y_train, y_val = preprocessor.train_val_split(X_scaled, y)print(f"Train: {X_train.shape}, Val: {X_val.shape}")

## 4.1 多重共线性诊断 (VIF)

In [ ]:
vif_df = compute_vif(X_train)print("Variance Inflation Factor (high VIF > 10 indicates multicollinearity):")print(vif_df.to_string(index=False))

## 4.2 PCA降维处理

In [ ]:
X_train_pca, pca_model, explained_var = apply_pca(X_train, variance_threshold=0.95)X_val_pca = pd.DataFrame(    pca_model.transform(X_val),    columns=[f"PC{i+1}" for i in range(X_train_pca.shape[1])],    index=X_val.index)print(f"Original features: {X_train.shape[1]}")print(f"PCA components retained: {X_train_pca.shape[1]}")print(f"Explained variance ratio: {explained_var}")print(f"Total explained variance: {explained_var.sum():.4f}")

## 4.3.1 普通最小二乘法 (OLS)

In [ ]:
lr = BlueberryLinearRegression()lr.fit_ols(X_train_pca, y_train)y_pred_ols = lr.predict(X_val_pca)metrics_ols = lr.evaluate(y_val, y_pred_ols)print("OLS Metrics:")for k, v in metrics_ols.items():    print(f"  {k}: {v:.4f}")print(f"\nIntercept: {lr.get_intercept():.4f}")

## 4.3.2 岭回归 (Ridge)

In [ ]:
lr.fit_ridge(X_train_pca, y_train, alpha=1.0)y_pred_ridge = lr.predict(X_val_pca)metrics_ridge = lr.evaluate(y_val, y_pred_ridge)print("Ridge Metrics:")for k, v in metrics_ridge.items():    print(f"  {k}: {v:.4f}")print(f"\nCoefficients:")print(lr.get_coefficients(X_train_pca.columns.tolist()))

## 4.3.3 Lasso回归

In [ ]:
lr.fit_lasso(X_train_pca, y_train, alpha=0.1)y_pred_lasso = lr.predict(X_val_pca)metrics_lasso = lr.evaluate(y_val, y_pred_lasso)print("Lasso Metrics:")for k, v in metrics_lasso.items():    print(f"  {k}: {v:.4f}")

## 4.4 残差分析（以Ridge为例）

In [ ]:
plot_residuals(y_val.values, y_pred_ridge)plot_actual_vs_predicted(y_val.values, y_pred_ridge, "Linear Regression (Ridge)")